In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
file_path = "/content/tesla_deliveries_dataset_2015_2025.csv"
df = pd.read_csv(file_path)

In [ ]:
df.shape[0],df.shape[1]


(2640, 12)

In [ ]:
df.head(5)

,Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
0,2023,5,Europe,Model S,17646,17922,92874.27,120,704,1863.42,Interpolated (Month),12207
1,2015,2,Asia,Model X,3797,4164,62205.65,75,438,249.46,Official (Quarter),7640
2,2019,1,North America,Model X,8411,9189,117887.32,82,480,605.59,Interpolated (Month),14071
3,2021,2,North America,Model 3,6555,7311,89294.91,120,712,700.07,Official (Quarter),9333
4,2016,12,Middle East,Model Y,12374,13537,114846.78,120,661,1226.88,Estimated (Region),8722


In [ ]:
df_info = pd.DataFrame({
    "Data Type": df.dtypes,
    "Non-Null Count": df.notnull().sum(),
    "Missing Values": df.isnull().sum()

})
print(df_info)

                     Data Type  Non-Null Count  Missing Values
Year                     int64            2640               0
Month                    int64            2640               0
Region                  object            2640               0
Model                   object            2640               0
Estimated_Deliveries     int64            2640               0
Production_Units         int64            2640               0
Avg_Price_USD          float64            2640               0
Battery_Capacity_kWh     int64            2640               0
Range_km                 int64            2640               0
CO2_Saved_tons         float64            2640               0
Source_Type             object            2640               0
Charging_Stations        int64            2640               0


In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Estimated_Deliveries', axis=1)
y = df['Estimated_Deliveries']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# separate columns by data types
numeric_features = ['Year', 'Month', 'Production_Units', 'Avg_Price_USD', 'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons', 'Charging_Stations']
categorical_features = ['Region', 'Model', 'Source_Type']

# Scaling
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# One-hot encoding
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combine
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print(len(numeric_features))
print(len(categorical_features))

8
3


In [ ]:
from xgboost import XGBRegressor

# end to end pipeline
pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('regressor', XGBRegressor(n_estimators=100, learning_rate =0.1, random_state=42))])

pipeline.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['Year', 'Month',
                                                   'Production_Units',
                                                   'Avg_Price_USD',
                                                   'Battery_Capacity_kWh',
                                                   'Range_km', 'CO2_Saved_tons',
                                                   'Charging_Stations']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Region', 'Model',
                                                   'Source_Type'])]...
                              feature_types=None, feature_weights=None,
                              gamma=None, grow_policy=None,
                              importance_type=None,
                              interaction_constraints=None, learning_rate=0.1,
                              max_bin=None, max_cat_threshold=None,
                              max_cat_to_onehot=None, max_delta_step=None,
                              max_depth=None, max_leaves=None,
                              min_child_weight=None, missing=nan,
                              monotone_constraints=None, multi_strategy=None,
                              n_estimators=100, n_jobs=None,
                              num_parallel_tree=None, ...))])

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

In [ ]:
print(f"Mean Absolute Error: {mae}")
print(f"Mean Squared Error: {mse}")
print(f"Root Mean Squared Error: {rmse}")
print(f"R-squared: {r2}")

Mean Absolute Error: 252.0694580078125
Mean Squared Error: 108959.0078125
Root Mean Squared Error: 330.08939366859397
R-squared: 0.99269038438797


In [ ]:
# Comparison betwwn actual and predicted
comparison_table = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred})
print(comparison_table)

      Actual     Predicted
2005    6991   7501.699707
32      9326   9392.213867
962     9061   8935.174805
1461    8951   9445.270508
478     8707   8899.014648
...      ...           ...
2033   18572  20168.617188
279     1761   1892.114136
2443   10136  10658.626953
2356   17905  17761.671875
1793   10489  10470.051758

[528 rows x 2 columns]


In [ ]:
# plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6, color='purple')

plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], linestyle='--', color='red', lw=2)
plt.title('Actual vs. Predicted Values')
plt.xlabel('Actual Values')
plt.ylabel('Predicted Values')
plt.grid(True)
plt.show()
